## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [2]:
load_dotenv(override=True)

True

In [3]:
# Constants 

MODEL_NAME = "gpt-5.4-nano"
USE_EMAIL = False
HOW_MANY_SEARCHES = 5

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [4]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [5]:
## Gratis alternatief

from duckduckgo_search import DDGS
from ddgs import DDGS
# Zorg dat je function_tool importeert uit de agents-bibliotheek van Ed Donner
from agents import Agent, Runner, ModelSettings, function_tool

# 1. Voeg hier de @function_tool decorator toe!
@function_tool
def web_search(query: str) -> str:
    """Searches the web using DuckDuckGo for free and returns top results."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No search results found."
        
        snippets = []
        for r in results:
            snippets.append(f"Title: {r.get('title')}\nURL: {r.get('href')}\nSnippet: {r.get('body')}\n")
        return "\n---\n".join(snippets)

# 2. Original setup with minimal changes
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""

task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")

# Nu snapt het framework wel dat dit een geldige tool is
tools = [web_search]

search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

Across 2026 “best/most popular” lists, the most frequently cited AI agent frameworks cluster around a few widely adopted ecosystems: **LangGraph/LangChain** (LangGraph especially for graph-based multi-step agent workflows), **CrewAI** (coordinating roles/agents for task completion), and **Microsoft’s agent tooling** (often framed around the Microsoft Agent Framework and related orchestration patterns). Other commonly mentioned frameworks include **OpenAI’s Agents SDK** for building agentic apps with the OpenAI model stack, and **AutoGen** for multi-agent conversations and tool-using workflows. Several guides also call out **PydanticAI** and **Mastra** as popular options for strongly-typed agent development and production-oriented orchestration.

In terms of why these stand out, the recurring evaluation criteria are **production readiness** (deployment patterns, reliability, and safety), **orchestration flexibility** (single-agent vs multi-agent, state management, and workflow graphs), and **observability** (tracing/debugging of tool calls and agent steps). Many 2026 writeups emphasize that practitioners choose frameworks based on workflow shape: graph/stateful orchestration (LangGraph), role-based coordination (CrewAI), conversational/multi-agent simulation (AutoGen), or SDK-style integration with a specific model provider (OpenAI Agents SDK). Overall, coverage suggests these frameworks are the de facto “top picks” in 2026 due to community adoption, active maintenance, and strong alignment with real-world agent tooling needs.

In [6]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [7]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

In 2026, the most widely adopted AI agent frameworks cluster around orchestration/state management, multi-agent “roles,” and production tooling. Across recent comparisons, **LangGraph** is repeatedly highlighted for building **complex, production-grade agent workflows** using explicit **state graphs** and controllable execution paths. **CrewAI** is commonly cited for enabling faster **multi-agent prototypes** via an intuitive, role-based “crew” abstraction. **Microsoft AutoGen** (often discussed alongside **Semantic Kernel**) is also frequently mentioned for **collaborative multi-agent conversations** and enterprise integration patterns.

The broader ecosystem also surfaces repeatedly: **LangChain** remains popular as a general-purpose foundation for chaining tools and models, while **LlamaIndex** is commonly included for agent workflows grounded in **retrieval-augmented generation (RAG)**. Some lists specifically note frameworks emphasizing **observability, evaluation, and deployment readiness**, with **enterprise choices** leaning toward Microsoft-oriented tools when teams are heavily invested in the Azure stack.

Overall, “most popular” in practice tends to mean a mix of community momentum (stars/content/examples), ease of getting started (especially for multi-agent role setups), and fit for production needs (stateful orchestration, tooling for testing/evaluation, and integration with LLM providers and vector/RAG components).

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
# Altijd eerst reason en dan de query
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [9]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [10]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [12]:
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Identify the most widely adopted AI agent frameworks and platforms mentioned in 2026-focused tech roundups.', query='2026 most popular AI agent framework'), WebSearchItem(reason='Find analyst or community-driven lists/rankings that reflect popularity and usage trends up to 2026.', query='2026 AI agent framework landscape popular frameworks'), WebSearchItem(reason='Determine which open-source agent frameworks (and libraries) are trending and frequently used by developers in 2026.', query='GitHub stars trending AI agent framework 2026 LangGraph CrewAI AutoGen'), WebSearchItem(reason='Cover major commercial/hosted agent platforms that may be considered frameworks and are popular in 2026.', query='2026 hosted AI agent platform framework Microsoft Azure Google AWS agent'), WebSearchItem(reason='Look for recent benchmarks, tutorials, and adoption indicators comparing leading agent frameworks.', query='2026 comparison of AI agent frameworks LangGr

## Agent 3: The Writer Agent

In [13]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## Agent 4: The email agent

In [14]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [15]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [16]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [17]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [18]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Showtime!

In [19]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Push: Subject: Most Popular AI Agent Frameworks in 2026 (Evidence-Based Landscape Report)

Hi,

Below is a clean, evidence-based summary of the most popular AI agent frameworks in 2026, based on the provided landscape report.

Most popular frameworks (most consistently cited across 2026 coverage):
1) LangGraph (LangChain family) — graph-based orchestration and stateful control for complex agent workflows.
2) LlamaIndex — retrieval/data/indexing foundation for grounded agents.
3) CrewAI — role/team abstraction for multi-agent collaboration.
4) Microsoft AutoGen — agent-to-agent communication patterns and configurable multi-agent interactions.

Key signals noted in the report:
- Strongest quantified GitHub traction: LangGraph (~43k stars), LlamaIndex (~52–53k stars), AutoGen (~54k stars).
- CrewAI is repeatedly highlighted in 2026 comparis

In [20]:
from IPython.display import display, Markdown

# Bekijk de gegenereerde markdown van het rapport
display(Markdown(report.markdown_report))

# Most Popular AI Agent Frameworks in 2026: An Evidence-Based Landscape Report

## 1. Why “Most Popular” Is Hard to Measure (and What 2026 Evidence Actually Shows)
When people ask for the *most popular* AI agent frameworks, they’re often combining at least three different concepts of “popularity”:

1. **Community attention**: how frequently frameworks are mentioned in blog posts, tutorials, and comparison guides.
2. **Developer adoption**: how many teams actually use the framework in prototypes and production systems.
3. **Repository vitality**: GitHub/GitLab indicators like stars, active contributors, issue/PR velocity, and release cadence.

In the 2026 web material summarized for this report, most sources **do not provide standardized, survey-weighted adoption studies** or a consistent benchmark methodology. Instead, they rely on narrative claims such as “widely adopted,” “most used in enterprise,” or “common choice for agentic apps,” plus a set of practical evaluation criteria (tool calling, state management, orchestration patterns, observability, and integration maturity). Some articles *attempt* to use GitHub metrics, but even those are usually secondary and not universally standardized.

**Practical takeaway:** In 2026, the most defensible way to identify “most popular” frameworks is a *triangulation approach*:
- Cross-reference consistent mentions across multiple 2026 comparison pages.
- Validate with **objective repository signals** where available (stars, contributors).
- Confirm architectural fit to real-world “agentic” workloads (multi-step control flow, tool usage, retrieval grounding, monitoring).

With that framing, the remainder of this report consolidates the most commonly cited and best-supported frameworks in 2026.

---

## 2. The 2026 “Core Set” of Popular Agent Frameworks
Across the 2026 comparison coverage, a recurring cluster of frameworks repeatedly appears as leading options for building “agentic” applications:

1. **LangGraph / LangChain-family tooling**
2. **LlamaIndex**
3. **CrewAI**
4. **Microsoft AutoGen**

Although these frameworks are often grouped together under the umbrella of “agent frameworks,” they differ substantially in how they structure agent behavior:
- LangGraph emphasizes **graph-based orchestration** and **stateful, long-running workflows**.
- LlamaIndex emphasizes **data/connectors** and **retrieval/indexing** to ground agent outputs.
- CrewAI emphasizes **multi-agent “team/role” orchestration**.
- AutoGen emphasizes **agent-to-agent communication scaffolding** and configurable interaction patterns.

This diversity matters because “popularity” in developer ecosystems often follows *use-case fit* as much as it follows raw feature count.

---

## 3. LangGraph (LangChain-family): Graph/Stateful Orchestration as the Default for Complex Agents
### 3.1 What 2026 coverage says people use it for
Multiple 2026 articles and roundups position LangGraph (within the LangChain ecosystem) as a central choice for building complex agent workflows. The recurring justifications are:
- **Multi-step orchestration** beyond simple prompt-to-output.
- **State management** suitable for resilient workflows and long-running agent interactions.
- **Tool/function calling integration** patterns.
- **Workflow control primitives** often described as “stateful graphs vs simple chains.”

In other words, LangGraph is commonly treated as the orchestration backbone for when a “chain of calls” needs to become a robust application flow.

### 3.2 Popularity signals in 2026
From the provided research summary, LangGraph has strong GitHub-style traction: **tens of thousands of stars** (e.g., ~**43k stars**) and a sizeable contributor base (e.g., ~**291 contributors** in the Star History-style tracker reference).

Even though stars are not a direct measure of “enterprise usage,” they are a useful proxy for:
- community interest,
- iteration velocity,
- and ecosystem size.

### 3.3 Why it tends to be considered “most popular”
LangGraph’s recurring advantage is that it addresses a common pain point in agent production: as agents become more complex, you need **control flow**, **intermediate state**, **retries**, **branching**, and **tool execution supervision**.

Comparison guides in 2026 frequently describe LangGraph as the “go-to” option when you need:
- deterministic orchestration patterns,
- more transparent state transitions,
- and easier adaptation from experimentation to production.

---

## 4. LlamaIndex: Retrieval/Data Infrastructure as a First-Class Agent Component
### 4.1 What 2026 coverage says people use it for
In 2026 comparisons, LlamaIndex is repeatedly framed as a leading option for **data/knowledge-indexing** and **retrieval-centric agent workflows**. The recurring narrative:
- LlamaIndex helps build agents that can **ground** responses in knowledge sources.
- It offers connectors, indexing, and retrieval/query abstractions.
- It is frequently selected when agent behavior must be anchored in external corpora (docs, databases, knowledge bases, etc.).

This distinguishes it from orchestration-first frameworks. Many teams don’t just need an agent to call tools; they need the agent to **find the right knowledge** reliably.

### 4.2 Popularity signals in 2026
In the summarized research, LlamaIndex appears even stronger in star/contributor indicators, reportedly around **~52–53k stars** with **~2k contributors**.

Again, while GitHub stars can’t fully quantify production deployment, they correlate with:
- the existence of a broad developer community,
- frequent issue/PR activity,
- and long-term maintenance.

### 4.3 Why it remains a “top popular” framework
Because many real agent applications are retrieval-augmented (RAG) or knowledge-grounded, a framework that excels at indexing/connectors/retrieval is frequently adopted broadly. In 2026 comparison writeups, LlamaIndex is often named as a “must-consider” component for teams building grounded assistants.

---

## 5. CrewAI: Role/Team Abstractions for Multi-Agent Applications
### 5.1 What 2026 coverage says people use it for
CrewAI is highlighted in 2026 sources as a multi-agent framework geared toward **role-based or team-style orchestration**. The typical positioning:
- quick-to-ship multi-agent setups,
- explicit “agents with roles” modeling,
- team-based collaboration patterns.

Compared to AutoGen (interaction scaffolding) or LangGraph (stateful control), CrewAI is often described as more “application-centric” for teams that want to assemble multiple roles quickly.

### 5.2 Popularity signals (qualitative)
The provided research summary stresses that many 2026 comparison articles name CrewAI as a leading multi-agent framework and that GitHub metrics are strong across the listed ecosystems. However, unlike LangGraph, the summary does not provide a specific CrewAI star/contributor number.

Still, the consistent presence of CrewAI in multiple 2026 “top frameworks” lists suggests it had notable mindshare during the year.

### 5.3 Why it can be “most popular” despite less repo-specific data here
Multi-agent systems are notoriously complex to design. Developers often seek abstractions that:
- reduce configuration burden,
- clarify responsibilities,
- and make it easier to visualize or reason about agent interactions.

CrewAI’s team/role abstraction aligns with how many teams naturally conceptualize agent workflows (“researcher,” “planner,” “writer,” “critic,” etc.). That alignment often drives adoption and community discussion—one of the key drivers of perceived popularity.

---

## 6. Microsoft AutoGen: Agent-to-Agent Communication Patterns (and Flexible Conversation Flows)
### 6.1 What 2026 coverage says people use it for
AutoGen is framed in 2026 sources as a framework focused on **agent-to-agent conversation/orchestration** patterns. In particular, coverage emphasizes:
- multi-agent conversation scaffolding,
- configurable interaction flows,
- flexible patterns that can support research-style iteration.

Unlike frameworks centered primarily on application state graphs, AutoGen is often treated as a communication/orchestration scaffold for agent interaction.

### 6.2 Popularity signals in 2026
The summarized research reports AutoGen crossing **~54k GitHub stars** by early/mid-Feb 2026 and notes it is described as being in **maintenance mode**.

That last detail is important: “maintenance mode” can mean different things. It might indicate that core functionality is stable and mature, which can still support popularity because teams prefer dependable tooling. Even if new features slow, the framework can remain widely used.

### 6.3 Why it remains among the most popular
AutoGen’s architecture fits a common developer goal: “Make multiple agents talk to each other in controllable ways.” For many use cases—especially iterative multi-agent experimentation—such scaffolding is more valuable than a higher-level team metaphor or a graph-only workflow.

---

## 7. Additional Frameworks Mentioned in 2026 (and Why They’re Often Second-Tier in “Top Popular” Lists)
The provided summarized search results emphasize that 2026 coverage generally avoids hard rankings but frequently lists a **mix** of orchestration and agent tooling. Beyond the core set, commonly mentioned items include:

- **Other LangChain-adjacent tooling** (within the same ecosystem family)
- **Additional vendor or SDK “agents” offerings** from major model providers
- Concepts and connectors associated with **MCP / A2A-style integrations** in some landscape pages

However, these are less consistently named as “the most popular frameworks” across sources (or they appear primarily in vendor-oriented or consultancy-style guides). In 2026, the most durable popularity tends to attach to frameworks that:
- have sustained open-source community activity,
- provide clear orchestration primitives,
- and integrate cleanly with common model-provider APIs.

---

## 8. 2026 Selection Criteria: What People Evaluate When Choosing an Agent Framework
Across the 2026 comparison coverage, the reasons for choosing a framework tend to cluster into a few recurring dimensions.

### 8.1 Orchestration depth
- Is it a simple chain, or can you build complex multi-step flows?
- Can it handle branching, looping, and long-running tasks?

LangGraph typically scores well here (graph/stateful workflows).

### 8.2 Tool/function calling and workflow control
Popular frameworks are expected to support:
- structured tool calls,
- robust handoffs between LLM reasoning and tool execution,
- supervision mechanisms to handle failures or retries.

### 8.3 Memory/state handling
For production readiness, frameworks must support:
- explicit state modeling,
- durable conversation/workflow memory,
- safe recovery from partial failure.

### 8.4 Retrieval grounding and data connectors
LlamaIndex is frequently positioned as the “data backbone” for retrieval-centric agent workflows.

### 8.5 Observability and evaluation (the 2026 obsession)
Multiple sources stress evaluation and observability:
- tracing agent steps,
- logging tool calls,
- capturing failures and recovery events,
- monitoring cost/performance and reliability.

This focus aligns with a maturity shift in 2026: agent prototypes are common, but production debugging and reliability engineering are harder.

### 8.6 Ecosystem fit
Popular choices typically align well with:
- existing application stacks (Python vs JS/TS),
- integration patterns with model providers,
- ability to deploy in real systems.

---

## 9. Practical Conclusions: “Most Popular” in 2026 Looks Like a Multi-Lens Answer
Based on the summarized 2026 research, the most defensible statement is:

- **LangGraph/LangChain-family tooling** is most often described as the leading general-purpose orchestration framework for complex, stateful agent workflows.
- **LlamaIndex** is the most consistently highlighted framework for retrieval/data-centric agent systems.
- **CrewAI** is a frequent top pick for multi-agent “team/roles” implementations.
- **Microsoft AutoGen** is a recurring leading option for agent-to-agent conversation scaffolding and multi-agent interaction patterns, with strong repo traction.

Because “most popular” is not measured uniformly, the list above reflects a combination of:
- repeated prominence in 2026 landscape/roundup pages,
- and strong or at least notable community activity signals (especially for LangGraph, LlamaIndex, and AutoGen as indicated by star/contributor references).

---

## 10. How to Verify Popularity More Rigorously (If You Need a “Ranking”)
If your goal is truly to rank frameworks rather than list them, 2026-era qualitative writeups suggest you should build an evidence table using objective indicators, such as:

1. **GitHub/GitLab:** stars growth rate, contributors over time, PR/issue throughput.
2. **Package indexes:** PyPI/NPM download trends (where applicable).
3. **Release cadence:** how often stable releases occur.
4. **Ecosystem integrations:** availability of integrations, templates, and examples.
5. **Operational maturity signals:** documentation quality, evaluation/observability tooling, and production deployment guides.
6. **Job postings and enterprise mentions:** as a proxy for real adoption.

This approach directly addresses the summarized research caution: narrative comparison articles rarely offer consistent methodology or standardized benchmarks.

---

## 11. Bottom-Line Answer (2026)
**Most popular AI agent frameworks in 2026 (most consistently cited across 2026 coverage):**

1. **LangGraph (LangChain family)** — graph-based orchestration and stateful control for complex agent workflows.
2. **LlamaIndex** — retrieval/data/indexing foundation for grounded agents.
3. **CrewAI** — role/team abstraction for multi-agent collaboration.
4. **Microsoft AutoGen** — agent-to-agent communication patterns and configurable multi-agent interactions.

**Popularity evidence strength (from the provided research signals):**
- **Strongest quantified GitHub traction signals** were reported for **LangGraph (~43k stars)**, **LlamaIndex (~52–53k stars)**, and **AutoGen (~54k stars)**.
- **CrewAI** shows up repeatedly in comparison narratives as a leading option, though specific star/contributor numbers weren’t provided in the summary.

---

## Follow-up Questions (for deeper verification)
1. Do you want “popular” defined primarily by **GitHub activity**, **downloads**, **enterprise adoption**, or **community mindshare**?
2. Should the list be restricted to **open-source frameworks only**, or include **vendor agent SDKs** and “Agents” offerings from major model providers?
3. Are you focused on **single-agent tool use**, **retrieval-grounded agents**, **multi-agent systems**, or **graph/stateful workflows**?
4. Do you need a **ranked table** with a scoring rubric (stars growth, release cadence, documentation quality, observability features, etc.)?
5. Which language ecosystems matter most for your use case (**Python**, **JS/TS**, or both)?
6. Are you targeting production constraints such as **cost controls**, **human-in-the-loop**, **evaluation harnesses**, and **monitoring/alerting**?

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>